In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras import models, layers, optimizers

In [2]:
# 1. Data Loading and Sorting
df = pd.read_excel("price_data.xlsx", header = 0)
df = df.sort_values('year')

df

,year,avgTemp,minTemp,maxTemp,rainFall,avgPrice
0,20100101,-4.9,-11.0,0.9,0.0,2123
1,20100102,-3.1,-5.5,5.5,0.8,2123
2,20100103,-2.9,-6.9,1.4,0.0,2123
3,20100104,-1.8,-5.1,2.2,5.9,2020
4,20100105,-5.2,-8.7,-1.8,0.7,2060
...,...,...,...,...,...,...
2917,20171227,-3.9,-8.0,0.7,0.0,2865
2918,20171228,-1.5,-6.9,3.7,0.0,2884
2919,20171229,2.9,-2.1,8.0,0.0,2901
2920,20171230,2.9,-1.6,7.1,0.6,2901


In [3]:
# 2. Data Splitting
split_idx = int(len(df) * 0.8)
train_df = df.iloc[:split_idx]
test_df = df.iloc[split_idx:]

x_train = train_df[['avgTemp', 'rainFall', 'avgPrice']]
y_train = train_df[['avgPrice']]

# 3. Normalization (Fit on training data only)
x_scaler = MinMaxScaler()
x_train_scaled = x_scaler.fit_transform(x_train)

y_scaler = MinMaxScaler()
y_train_scaled = y_scaler.fit_transform(y_train)

print (len(df))
print (len(train_df))
print (len(test_df))

2922
2337
585


In [4]:
# Transform test data using the training data scaler
# [Important] Prepend the last 'seq_length' days of training data to the test set 
# to enable prediction for the very first timestamp of the test set.

last_train_days = train_df.tail(30) # Assuming seq_length is 30
combined_test_df = pd.concat([last_train_days, test_df], axis=0)

x_test = combined_test_df[['avgTemp', 'rainFall', 'avgPrice']]
y_test = combined_test_df[['avgPrice']]


x_test_scaled = x_scaler.transform(x_test)
y_test_scaled = y_scaler.transform(y_test)

x_test_scaled

array([[0.68304668, 0.01465969, 0.30926924],
       [0.57002457, 0.2617801 , 0.30140497],
       [0.5995086 , 0.02513089, 0.29734028],
       ...,
       [0.33660934, 0.        , 0.15975965],
       [0.33660934, 0.00628272, 0.15975965],
       [0.31695332, 0.00418848, 0.15975965]])

In [5]:
# 4. Time Series Dataset Building Function
def build_dataset(X, y, seq_length):
    dataX, dataY = [], []
    for i in range(len(X) - seq_length):
        dataX.append(X[i : i + seq_length])
        dataY.append(y[i + seq_length])
    return np.array(dataX), np.array(dataY)

seq_length = 30
trainX, trainY = build_dataset(x_train_scaled, y_train_scaled, seq_length)
testX, testY = build_dataset(x_test_scaled, y_test_scaled, seq_length)

# 5. Model Architecture (Functionalized for comparison)
def create_model(model_type='LSTM'):
    model = models.Sequential()
    if model_type == 'LSTM':
        model.add(layers.LSTM(64, input_shape=(seq_length, 3), return_sequences=True))
    else:
        model.add(layers.SimpleRNN(64, input_shape=(seq_length, 3), return_sequences=True))
    
    model.add(layers.Dropout(0.2))
    model.add(layers.LSTM(32) if model_type == 'LSTM' else layers.SimpleRNN(32))
    model.add(layers.Dense(1))
    model.compile(optimizer='adam', loss='mse')
    return model

In [ ]:
# 6. Training and Prediction
lstm_model = create_model('LSTM')
rnn_model = create_model('RNN')

print("Training LSTM Model...")
lstm_model.fit(trainX, trainY, epochs=30, batch_size=16, verbose=1)
print("Training RNN Model...")
rnn_model.fit(trainX, trainY, epochs=30, batch_size=16, verbose=1)

# 7. Inverse Transformation and Result Analysis
lstm_pred = y_scaler.inverse_transform(lstm_model.predict(testX))
rnn_pred = y_scaler.inverse_transform(rnn_model.predict(testX))
actual = y_scaler.inverse_transform(testY)

C:\Users\oem\.conda\envs\aifoodtech\lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Training LSTM Model...
Epoch 1/30
145/145 ━━━━━━━━━━━━━━━━━━━━ 5s 21ms/step - loss: 0.0074
Epoch 2/30
145/145 ━━━━━━━━━━━━━━━━━━━━ 3s 22ms/step - loss: 0.0013
Epoch 3/30
145/145 ━━━━━━━━━━━━━━━━━━━━ 3s 22ms/step - loss: 8.1212e-04
Epoch 4/30
145/145 ━━━━━━━━━━━━━━━━━━━━ 3s 23ms/step - loss: 7.2741e-04
Epoch 5/30
145/145 ━━━━━━━━━━━━━━━━━━━━ 3s 22ms/step - loss: 6.0507e-04
Epoch 6/30
145/145 ━━━━━━━━━━━━━━━━━━━━ 3s 22ms/step - loss: 5.5255e-04
Epoch 7/30
145/145 ━━━━━━━━━━━━━━━━━━━━ 3s 22ms/step - loss: 7.5368e-04
Epoch 8/30
145/145 ━━━━━━━━━━━━━━━━━━━━ 3s 23ms/step - loss: 5.0796e-04
Epoch 9/30
145/145 ━━━━━━━━━━━━━━━━━━━━ 3s 21ms/step - loss: 4.9734e-04
Epoch 10/30
145/145 ━━━━━━━━━━━━━━━━━━━━ 3s 22ms/step - loss: 5.3279e-04
Epoch 11/30
145/145 ━━━━━━━━━━━━━━━━━━━━ 3s 20ms/step - loss: 4.4840e-04
Epoch 12/30
145/145 ━━━━━━━━━━━━━━━━━━━━ 3s 21ms/step - loss: 3.8845e-04
Epoch 13/30
145/145 ━━━━━━━━━━━━━━━━━━━━ 3s 20ms/step - loss: 3.7529e-04
Epoch 14/30
145/145 ━━━━━━━━━━━━━━━━━━━━ 3s 2

In [ ]:
# 8. Visualization
plt.figure(figsize=(14, 6))
full_actual_scaled = np.concatenate([trainY, testY], axis=0)
full_actual = y_scaler.inverse_transform(full_actual_scaled)
plt.plot(full_actual, label='Full Actual Price', color='royalblue', alpha=0.6, linewidth=2)
plt.title("Full Actual Cabbage Price|")
plt.show()

plt.figure(figsize=(14, 6))
plt.plot(actual, label='Actual Price', color='black', linewidth=1.5)
plt.plot(lstm_pred, label='LSTM (Standard)', color='red', linestyle='--')
plt.plot(rnn_pred, label='RNN (Standard)', color='green', linestyle='-.')
plt.title("Cabbage Price Prediction")
plt.legend()
plt.show()

In [ ]:
# Print error comparison
lstm_rmse = np.sqrt(np.mean((actual - lstm_pred)**2))
rnn_rmse = np.sqrt(np.mean((actual - rnn_pred)**2))
print(f"\n[Result Analysis]")
print(f"LSTM RMSE: {lstm_rmse:.2f} Won")
print(f"RNN RMSE: {rnn_rmse:.2f} Won")